<a href="https://colab.research.google.com/github/kosar-am/finance-prediction-rnn/blob/main/notebooks/02_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Google Stock Price Prediction using RNN, LSTM, and GRU

## 02 — Data Preprocessing

In this notebook, the raw Google stock data will be prepared for deep learning models through preprocessing steps such as feature selection, scaling, train-test splitting, and sequence generation.

### Objectives

- Load the raw dataset
- Select the target feature
- Split the dataset into training and testing sets
- Scale the data using Min-Max normalization
- Create input sequences for recurrent neural networks
- Prepare the data for model training

## 📚 Import Required Libraries

In [1]:
import numpy as np
import pandas as pd

import yfinance as yf

from sklearn.preprocessing import MinMaxScaler

## 📥 Download Dataset

In [2]:
ticker = "GOOG"

df = yf.download(
    ticker,
    start="2015-01-01",
    end="2025-12-31",
    auto_adjust=True
)

[*********************100%***********************]  1 of 1 completed


## 🧹 Clean Column Names

In [3]:
# Remove the MultiIndex column structure
df.columns = df.columns.get_level_values(0)
df.columns.name = None

df.head()

,Close,High,Low,Open,Volume
Date,,,,,
2015-01-02,25.939949,26.259251,25.904856,26.147544,28951268
2015-01-05,25.399218,25.916228,25.359182,25.863340,41196796
2015-01-06,24.810535,25.513146,24.765558,25.455068,57998800
2015-01-07,24.768030,25.071711,24.696360,25.059650,41301082
2015-01-08,24.846121,24.885662,24.268810,24.614307,67071641


## 🎯 Feature Selection

In [4]:
# Select the closing price as the target feature
data = df[["Close"]].copy()

data.head()

,Close
Date,
2015-01-02,25.939949
2015-01-05,25.399218
2015-01-06,24.810535
2015-01-07,24.768030
2015-01-08,24.846121


In [5]:
data.shape

(2765, 1)

In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2765 entries, 2015-01-02 to 2025-12-30
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Close   2765 non-null   float64
dtypes: float64(1)
memory usage: 43.2 KB


### Observation

The dataset now contains only the closing price as the target feature. The data remains complete, numerical, and chronologically ordered.

### Insight

The selected feature is now ready for time-series preprocessing, including train-test splitting and normalization.

## ✂️ Train-Test Split

In [7]:
# Define the training data size
train_size = int(len(data) * 0.8)

# Split the data chronologically
train_data = data.iloc[:train_size]
test_data = data.iloc[train_size:]

In [8]:
print("Training data shape:", train_data.shape)
print("Testing data shape:", test_data.shape)

Training data shape: (2212, 1)
Testing data shape: (553, 1)


### Observation

The dataset was split chronologically into training and testing sets using an 80:20 ratio. The training set contains the earlier observations, while the testing set contains the most recent data.


### Insight

Maintaining the chronological order prevents data leakage and ensures that the models are evaluated on unseen future data, which reflects real-world stock price forecasting.

## 📏 Feature Scaling

In [9]:
scaler = MinMaxScaler()

# Learn from the training data
scaler.fit(train_data)

# Apply to the training data
train_scaled = scaler.transform(train_data)

# Apply to the testing data
test_scaled = scaler.transform(test_data)

In [10]:
print(train_scaled[:5])

[[0.01275147]
 [0.00842723]
 [0.00371952]
 [0.0033796 ]
 [0.0040041 ]]


In [11]:
print(test_scaled[:5])

[[0.92295611]
 [0.90940072]
 [0.90702246]
 [0.88926587]
 [0.89846106]]


In [12]:
print(train_scaled.min(), train_scaled.max())

0.0 1.0


## 🪟 Create Sequences

In [13]:
# Define the sequence length
sequence_length = 60

In [14]:
def create_sequences(data, sequence_length):
    X = []
    y = []

    for i in range(sequence_length, len(data)):
        X.append(data[i - sequence_length:i])
        y.append(data[i])

    return np.array(X), np.array(y)

In [15]:
X_train, y_train = create_sequences(
    train_scaled,
    sequence_length
)

In [16]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (2152, 60, 1)
y_train shape: (2152, 1)


In [17]:
X_test, y_test = create_sequences(
    test_scaled,
    sequence_length
)

In [18]:
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_test shape: (493, 60, 1)
y_test shape: (493, 1)


### Observation

The training and testing datasets were successfully converted into sequential samples using a sliding window approach with a sequence length of 60. Each input sample contains 60 consecutive closing prices, while the corresponding target represents the closing price of the next trading day.


### Insight

Creating sequential input samples enables recurrent neural networks to learn temporal dependencies from historical stock prices. This preprocessing step transforms the raw time series into a supervised learning dataset, making it suitable for training RNN, LSTM, and GRU models.